## ACUTE MYELOID LUEKANEMIA PHENOTYPE FILE 
This script creates the phenotype file for SAIGE-GENE+ with the deprviation index. It integrates several inputs, including EUR-like exome sample quality control results, ancestry-specific PCs, and age and sex files. Only individuals who pass exome quality control are retained in the SAIGE-GENE+ phenotype file. Case–control status is assigned by providing a list of person IDs for individuals designated as cases; all remaining individuals are assigned as controls. The script also restricts the dataset to individuals who have electronic health record data. All quantitative covariates including the deprivatrion index are standardized to have a mean of 0 and a standard deviation of 1. Because more samples passed exome quality control than ACAF quality control in this analysis, this phenotype file will also be used for the ACAF analysis, with samples failing ACAF quality control excluded at a later stage.

In [1]:
# Load packages 
from sklearn.preprocessing import StandardScaler
from functools import reduce
import pandas as pd 
import os 

In [2]:
# Workspace bucket 
my_bucket = os.getenv("WORKSPACE_BUCKET")

In [3]:
# Make directory for aml phenotypes files 
!rm -rf aml_phenotype_files
!mkdir -p aml_phenotype_files

In [4]:
# Copy quality control information and demographic information to workspace 
!gsutil cp {my_bucket}/data/pcs/EUR_flashpca.pcs aml_phenotype_files
!gsutil cp {my_bucket}/data/covariates/observation_age_period.tsv aml_phenotype_files
!gsutil cp {my_bucket}/data/wgs_qc/analysis_samples_wgs_qc/exome/EUR-like_exome_qc.txt aml_phenotype_files
!gsutil cp {my_bucket}/data/wgs_qc/ancestry_sd3_wgs_qc/EUR-like_cohort_info_qc.txt aml_phenotype_files

Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/pcs/EUR_flashpca.pcs...
- [1 files][ 64.3 MiB/ 64.3 MiB]                                                
Operation completed over 1 objects/64.3 MiB.                                     
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/covariates/observation_age_period.tsv...
- [1 files][ 29.0 MiB/ 29.0 MiB]                                                
Operation completed over 1 objects/29.0 MiB.                                     
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/wgs_qc/analysis_samples_wgs_qc/exome/EUR-like_exome_qc.txt...
/ [1 files][  2.1 MiB/  2.1 MiB]                                                
Operation completed over 1 objects/2.1 MiB.                                      
Copying gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/wgs_qc/ancestry_sd3_wgs_qc/EUR-like_cohort_info_qc.txt...
\ [1 files][139.2 MiB/139.2 MiB]                                  

In [5]:
# Load quality control information and demographic information 
eur_pca = pd.read_csv("aml_phenotype_files/EUR_flashpca.pcs", sep ='\t')
eur_exome = pd.read_csv("aml_phenotype_files/EUR-like_exome_qc.txt", sep ='\t')
age = pd.read_csv("aml_phenotype_files/observation_age_period.tsv", sep ='\t')
sex = pd.read_csv("aml_phenotype_files/EUR-like_cohort_info_qc.txt", sep ='\t')

In [6]:
# Assign sex variable 
sex['sex'] = sex['dragen_sex_ploidy'].apply(lambda x: 1 if x == "XX" else 0 if x == "XY" else 2)
sex['sex'] = sex['sex'].astype("int")

In [7]:
import pandas
import os

# This query represents dataset "aml_cohort" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_39009800_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (140352)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN (4304199) 
                    AND is_standard = 1 )) criteria ) )"""

aml_person_df = pandas.read_gbq(
    dataset_39009800_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

Downloading:   0%|          | 0/774 [00:00<?, ?rows/s]

In [8]:
import pandas
import os

# This query represents dataset "electronic_health_records" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_72216074_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) )"""

ehr_person_df = pandas.read_gbq(
    dataset_72216074_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

Downloading:   0%|          | 0/393601 [00:00<?, ?rows/s]

In [9]:
import pandas
import os

# This query represents dataset "zip_code_socioeconmic_cohort" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v8
dataset_01113255_zip_code_socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3 
            AND observation_source_concept_id = 1585250 
            AND observation.value_as_string NOT LIKE 'Res%'"""

zip_code_socioeconomic_df = pandas.read_gbq(
    dataset_01113255_zip_code_socioeconomic_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

zip_code_socioeconomic_df.head(5)

Downloading:   0%|          | 0/606414 [00:00<?, ?rows/s]

,person_id,observation_datetime,zip_code,assisted_income,high_school_education,median_income,no_health_insurance,poverty,vacant_housing,deprivation_index,american_community_survey_year
0,6240054,2023-01-15 19:25:22+00:00,847**,10.101987,92.145159,53574.806732,14.805154,15.248625,19.587874,0.338073,2017
1,1827774,2020-09-02 01:31:06+00:00,551**,11.941515,91.303142,72119.429796,5.744544,12.124588,4.147869,0.270221,2017
2,8195956,2022-03-18 23:55:09+00:00,945**,7.608938,89.297270,99512.603711,6.163447,8.640321,4.435022,0.231635,2017
3,5579825,2022-05-10 15:04:03+00:00,461**,7.983830,90.967591,66242.443830,8.252932,9.391962,7.415109,0.272563,2017
4,1321340,2017-09-25 15:03:43+00:00,919**,10.966441,82.405497,68904.984971,11.143829,13.856359,7.246927,0.321559,2017


In [10]:
# Create list of quality control information and demographic information 
dfs = [eur_exome.rename(columns={"IID":"person_id"}),
       age.rename(columns={"IID":"person_id"}).filter(regex="person_id|age"),
       sex.rename(columns={"IID":"person_id"}).filter(regex="person_id|sex"), 
       eur_pca.rename(columns={"IID":"person_id"}).filter(regex="person_id|PC"),
       ehr_person_df,
       zip_code_socioeconomic_df
       ]

In [11]:
# Combine list of quality control information and demographic information 
phenotype_file = reduce(
    lambda left, right: pd.merge(left, right, on="person_id", how="inner"),
    dfs
)

In [12]:
# Assign AML phenotype to individuals 
phenotype_file['aml_phenotype'] = phenotype_file['person_id'].isin(aml_person_df['person_id']).astype(int)

In [13]:
cases = phenotype_file[phenotype_file['aml_phenotype'] == 1]
# Median
median = cases['age'].median()

# Quartiles
Q1 = cases['age'].quantile(0.25)
Q3 = cases['age'].quantile(0.75)

# Interquartile Range
print("Median:", median)
print("Q1:", Q1)
print("Q3:", Q3)

controls = phenotype_file[phenotype_file['aml_phenotype'] == 0]
# Median
median = controls['age'].median()

# Quartiles
Q1 = controls['age'].quantile(0.25)
Q3 = controls['age'].quantile(0.75)

# Interquartile Range
print("Median:", median)
print("Q1:", Q1)
print("Q3:", Q3)

Median: 69.29500342231348
Q1: 58.2943189596167
Q3: 76.00273785078713
Median: 62.2943189596167
Q1: 46.2943189596167
Q3: 72.29568788501027


In [ ]:
# Format phenotype file  
phenotype_file = (phenotype_file[['aml_phenotype','sex', 'age', 
            'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 
            'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 
            'deprivation_index', 'assisted_income', 
            'high_school_education','median_income',
            'no_health_insurance', 'poverty',
            'vacant_housing', 
            'person_id']]
)

In [ ]:
# Identify columns that start with "PC" or "age"
cols_to_standardize = [col for col in phenotype_file.columns if col.startswith("PC") 
                       or col.startswith("age") 
                       or col.startswith("deprivation_index")
                       or col.startswith("assisted_income")
                       or col.startswith("high_school_education")
                       or col.startswith("median_income")
                       or col.startswith("no_health_insurance")
                       or col.startswith("poverty")
                       or col.startswith("vacant_housing")]
# Standardize only the selected columns
scaler = StandardScaler()
phenotype_file[cols_to_standardize] = scaler.fit_transform(phenotype_file[cols_to_standardize])
# Print results of the standarization 
print("============================================")
print("Check variances of standardized covariates:")
print("============================================")
print(phenotype_file.filter(regex="PC|age").var())
print(phenotype_file.describe())
print("============================================")

In [ ]:
# Final check for phenotype 
print("============================================")
print("Check the presence of aml phenotype:")
print("============================================")
print(phenotype_file["aml_phenotype"].value_counts())
print("============================================")

In [ ]:
# Save phenoytpe file
phenotype_file.to_csv("EUR_deprivation_index_file.txt", sep ='\t', index=None)

In [ ]:
# Save phenoytpe file to workspace bucket 
!gsutil cp EUR_deprivation_index_file.txt {my_bucket}/data/saige_phenotype_files/aml_phenotype_files/